## Get embeddings from dataset

This notebook gives an example on how to get embeddings from a large dataset.


#### 1. Load the dataset

The dataset used in this example is [fine-food reviews](https://www.kaggle.com/snap/amazon-fine-food-reviews) from Amazon. The dataset contains a total of 568,454 food reviews Amazon users left up to October 2012. We will use a subset of this dataset, consisting of 1,000 most recent reviews for illustration purposes. The reviews are in English and tend to be positive or negative. Each review has a ProductId, UserId, Score, review title (Summary) and review body (Text).

We will combine the review summary and review text into a single combined text. The model will encode this combined text and it will output a single vector embedding per review.

To run this notebook, you will need to install: pandas, openai, transformers, plotly, matplotlib, scikit-learn, torch (transformer dep), torchvision, and scipy.

In [2]:
import os
from dotenv import load_dotenv

# Load environment variables from the .env file in your directory
load_dotenv()

# Retrieve the API key from the environment
api_key = os.getenv("OPENAI_API_KEY")

# Verify if the key was loaded successfully
if api_key:
    print(f"✓ API Key loaded successfully. Starts with: {api_key[:7]}...")
else:
    print("✗ API Key not found. Please check your .env file.")

✓ API Key loaded successfully. Starts with: sk-proj...


In [3]:
import pandas as pd
import tiktoken
from typing import List, Optional

from openai import OpenAI
import numpy as np
import pandas as pd

client = OpenAI(max_retries=5)


#from utils.embeddings_utils import get_embedding

In [4]:
def get_embedding(text: str, model="text-embedding-3-small", **kwargs) -> List[float]:
    # replace newlines, which can negatively affect performance.
    text = text.replace("\n", " ")

    response = client.embeddings.create(input=[text], model=model, **kwargs)

    return response.data[0].embedding

In [5]:
embedding_model = "text-embedding-3-small"
embedding_encoding = "cl100k_base"
max_tokens = 8000  # the maximum for text-embedding-3-small is 8191

In [6]:
# load & inspect dataset
input_datapath = "data/fine_food_reviews_1k.csv"  # to save space, we provide a pre-filtered dataset
df = pd.read_csv(input_datapath, index_col=0)
df = df[["Time", "ProductId", "UserId", "Score", "Summary", "Text"]]
df = df.dropna()
df["combined"] = (
    "Title: " + df.Summary.str.strip() + "; Content: " + df.Text.str.strip()
)
df.head(2)

,Time,ProductId,UserId,Score,Summary,Text,combined
0,1351123200,B003XPF9BO,A3R7JR3FMEBXQB,5,where does one start...and stop... with a tre...,Wanted to save some to bring to my Chicago fam...,Title: where does one start...and stop... wit...
1,1351123200,B003JK537S,A3JBPC3WFUT5ZP,1,Arrived in pieces,"Not pleased at all. When I opened the box, mos...",Title: Arrived in pieces; Content: Not pleased...


In [7]:
# subsample to 1k most recent reviews and remove samples that are too long
top_n = 1000
df = df.sort_values("Time").tail(top_n * 2)  # first cut to first 2k entries, assuming less than half will be filtered out
df.drop("Time", axis=1, inplace=True)

encoding = tiktoken.get_encoding(embedding_encoding)

# omit reviews that are too long to embed
df["n_tokens"] = df.combined.apply(lambda x: len(encoding.encode(x)))
df = df[df.n_tokens <= max_tokens].tail(top_n)
len(df)

/home/maulik/miniconda3/envs/lang-chain/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


1000

#### 2. Get embeddings and save them for future reuse

In [8]:
# Ensure you have your API key set in your environment per README: https://github.com/openai/openai-python#usage

# This may take a few minutes
df["embedding"] = df.combined.apply(lambda x: get_embedding(x, model=embedding_model))
df.to_csv("data/fine_food_reviews_with_embeddings_1k.csv")

In [9]:
a = get_embedding("hi", model=embedding_model)

In [10]:
print(a)

[-0.00374603271484375, -0.0191650390625, 0.01221466064453125, 0.032745361328125, 0.016204833984375, -0.03717041015625, -0.02825927734375, 0.06494140625, 0.0009541511535644531, -0.057525634765625, 0.0014677047729492188, -0.03265380859375, -0.03076171875, 0.004787445068359375, 0.03289794921875, 0.019287109375, -0.04193115234375, -0.002918243408203125, 0.024688720703125, 0.046661376953125, 0.03778076171875, 0.0341796875, -0.00246429443359375, 0.0362548828125, 0.0163116455078125, 0.001827239990234375, 0.002094268798828125, -0.00887298583984375, 0.032073974609375, -0.0263214111328125, 0.0013322830200195312, -0.03778076171875, 0.0258331298828125, -0.037017822265625, -0.01654052734375, -0.0127105712890625, -0.025634765625, 0.03302001953125, 0.015533447265625, -0.037811279296875, 0.024993896484375, -0.00499725341796875, 0.046417236328125, 0.015869140625, -0.019866943359375, 0.015655517578125, -0.026947021484375, 0.0164031982421875, 0.003849029541015625, 0.02880859375, -0.0207061767578125, 0.00